# Hybrid DeBERTa + TF-IDF Ensemble
DeBERTa-v3-base (explanation-augmented, 2-fold CV) soft-voted with TF-IDF baseline (70/30).


In [ ]:
import subprocess
subprocess.run(
    [
        "pip", "install", "-q",
        "torch==2.2.2",
        "transformers", "accelerate", "evaluate", "scikit-learn",
    ],
    check=True,
)
print("deps ok")


In [ ]:
"""Hybrid DeBERTa-v3-base + TF-IDF soft-voting ensemble for llm-classification-finetuning."""

import glob
import json
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

OUTPUT_SUBMISSION = "/kaggle/working/submission.csv"
OUTPUT_FIG = "/kaggle/working/hybrid_label_distribution.png"
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LEN = 512
SEED = 42
DEBERTA_FOLDS = 2
TFIDF_FOLDS = 3
DEBERTA_EPOCHS = 2
DEBERTA_BATCH = 16
DEBERTA_LR = 2e-5
DEBERTA_TRAIN_SAMPLE = 12000
ENSEMBLE_WEIGHTS = {"deberta": 0.70, "tfidf": 0.30}
USE_EXPLANATION_PREFIX = True
SWAP_AUGMENT_TRAIN = True
SWAP_AVERAGE_INFERENCE = True


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
USE_CUDA = torch.cuda.is_available()
USE_FP16 = False
if USE_CUDA:
    try:
        _probe = torch.zeros(1, device="cuda")
        _probe = _probe + 1
        USE_FP16 = "p100" not in torch.cuda.get_device_name(0).lower()
        print("[env] gpu:", torch.cuda.get_device_name(0), "cuda_ok: True")
    except Exception as exc:
        USE_CUDA = False
        print("[env] cuda probe failed, falling back to CPU for DeBERTa:", exc)
print("[env] cuda:", USE_CUDA, "fp16:", USE_FP16)


def resolve_data_dir() -> str:
    need = {"train.csv", "test.csv", "sample_submission.csv"}
    for d in [
        "/kaggle/input/competitions/llm-classification-finetuning",
        "/kaggle/input/llm-classification-finetuning",
    ]:
        if os.path.isdir(d) and need.issubset(set(os.listdir(d))):
            return d
    for dirpath, _, filenames in os.walk("/kaggle/input"):
        if need.issubset(set(filenames)):
            return dirpath
    raise FileNotFoundError("Competition data not found — attach llm-classification-finetuning.")


DATA_DIR = resolve_data_dir()
print("[log] DATA_DIR", DATA_DIR)

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

for col in ["prompt", "response_a", "response_b"]:
    train_df[col] = train_df[col].fillna("").astype(str)
    test_df[col] = test_df[col].fillna("").astype(str)

print("[log] train_shape", train_df.shape, "test_shape", test_df.shape)


def winner_columns_to_label(df: pd.DataFrame) -> np.ndarray:
    cols = ["winner_model_a", "winner_model_b", "winner_tie"]
    w = df[cols].values.astype(np.float32)
    ok = np.isclose(w.sum(axis=1), 1.0, atol=1e-3)
    if not np.all(ok):
        raise ValueError("winner columns must sum to 1")
    return w.argmax(axis=1)


train_df["label"] = winner_columns_to_label(train_df)

vc = train_df["label"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["A", "B", "tie"], [vc.get(0, 0), vc.get(1, 0), vc.get(2, 0)])
ax.set_title("Train label distribution")
fig.tight_layout()
fig.savefig(OUTPUT_FIG, dpi=120)
plt.close(fig)


def build_heuristic_explanation(prompt: str, response_a: str, response_b: str) -> str:
    """Lightweight explanation prefix (SemEval-style) without a generative LLM."""
    words_a = response_a.split()
    words_b = response_b.split()
    shared = len(set(words_a) & set(words_b))
    len_a, len_b = len(words_a), len(words_b)
    notes = []
    if len_a > len_b * 1.15:
        notes.append("Response A is more detailed.")
    elif len_b > len_a * 1.15:
        notes.append("Response B is more detailed.")
    if shared >= 8:
        notes.append("Both responses share substantial wording.")
    if len(prompt.split()) > 120:
        notes.append("The prompt is long and may require careful comparison.")
    if not notes:
        notes.append("Compare clarity, correctness, and helpfulness.")
    return " ".join(notes)


def build_pair_text(prompt: str, response_a: str, response_b: str, swap: bool = False) -> str:
    if swap:
        response_a, response_b = response_b, response_a
    prefix = ""
    if USE_EXPLANATION_PREFIX:
        prefix = build_heuristic_explanation(prompt, response_a, response_b) + "\n\n"
    return (
        f"{prefix}Which response do judges prefer?\n\n"
        f"### Prompt\n{prompt}\n\n"
        f"### Response A\n{response_a}\n\n"
        f"### Response B\n{response_b}"
    )


def build_tfidf_text(df: pd.DataFrame, swap: bool = False) -> pd.Series:
    if not swap:
        return (
            "PROMPT: " + df["prompt"]
            + "\nRESPONSE_A: " + df["response_a"]
            + "\nRESPONSE_B: " + df["response_b"]
        )
    return (
        "PROMPT: " + df["prompt"]
        + "\nRESPONSE_A: " + df["response_b"]
        + "\nRESPONSE_B: " + df["response_a"]
    )


def softmax_np(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def swap_probs(probs: np.ndarray) -> np.ndarray:
    return probs[:, [1, 0, 2]]


def average_with_swap(probs: np.ndarray, probs_swapped: np.ndarray) -> np.ndarray:
    return (probs + swap_probs(probs_swapped)) / 2.0


# ── TF-IDF ensemble (baseline complement) ────────────────────────────────────
print("\n[stage] TF-IDF ensemble")
y = train_df["label"].values
X_text = build_tfidf_text(train_df)
X_test_text = build_tfidf_text(test_df)

tfidf_models = [
    Pipeline([
        ("tfidf", TfidfVectorizer(max_features=60000, ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=2000, C=2.0, n_jobs=-1)),
    ]),
    Pipeline([
        ("tfidf", TfidfVectorizer(max_features=70000, ngram_range=(1, 2), min_df=2)),
        ("clf", CalibratedClassifierCV(LinearSVC(C=1.0), cv=3)),
    ]),
]

skf_tfidf = StratifiedKFold(n_splits=TFIDF_FOLDS, shuffle=True, random_state=SEED)
tfidf_test_preds = []

for model_idx, model in enumerate(tfidf_models):
    fold_test = []
    for fold, (tr, va) in enumerate(skf_tfidf.split(X_text, y)):
        print(f"  tfidf model {model_idx + 1} fold {fold + 1}/{TFIDF_FOLDS}")
        model.fit(X_text.iloc[tr], y[tr])
        p_va = model.predict_proba(X_text.iloc[va])
        print(f"    val log_loss={log_loss(y[va], p_va):.5f}")

        p_test = model.predict_proba(X_test_text)
        if SWAP_AVERAGE_INFERENCE:
            p_swap = model.predict_proba(build_tfidf_text(test_df, swap=True))
            p_test = average_with_swap(p_test, p_swap)
        fold_test.append(p_test)
    tfidf_test_preds.append(np.mean(fold_test, axis=0))

tfidf_probs = np.mean(tfidf_test_preds, axis=0)
print("[log] tfidf_probs shape", tfidf_probs.shape)


# ── DeBERTa fine-tuning with explanation-augmented text ─────────────────────
print("\n[stage] DeBERTa-v3-small CV")
deberta_train_df = train_df
if len(train_df) > DEBERTA_TRAIN_SAMPLE:
    deberta_train_df = train_df.sample(n=DEBERTA_TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)
    print(f"[log] deberta subsample {len(deberta_train_df)} / {len(train_df)}")


class PairDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_len: int, is_test: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        text = build_pair_text(row["prompt"], row["response_a"], row["response_b"])
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_tensors=None,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

skf_deberta = StratifiedKFold(n_splits=DEBERTA_FOLDS, shuffle=True, random_state=SEED)
deberta_test_probs = np.zeros((len(test_df), 3), dtype=np.float64)
deberta_oof = np.zeros((len(deberta_train_df), 3), dtype=np.float64)
deberta_ok = True

try:
    for fold, (tr_idx, va_idx) in enumerate(skf_deberta.split(deberta_train_df, deberta_train_df["label"])):
        print(f"  deberta fold {fold + 1}/{DEBERTA_FOLDS}")
        fold_train = deberta_train_df.iloc[tr_idx].reset_index(drop=True)
        fold_val = deberta_train_df.iloc[va_idx].reset_index(drop=True)

        if SWAP_AUGMENT_TRAIN:
            swapped = fold_train.copy()
            swapped["response_a"] = fold_train["response_b"].values
            swapped["response_b"] = fold_train["response_a"].values
            swapped["label"] = fold_train["label"].map({0: 1, 1: 0, 2: 2}).values
            fold_train = pd.concat([fold_train, swapped], ignore_index=True)
            fold_train = fold_train.sample(frac=1, random_state=SEED + fold).reset_index(drop=True)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
        train_ds = PairDataset(fold_train, tokenizer, MAX_LEN, is_test=False)
        val_ds = PairDataset(fold_val, tokenizer, MAX_LEN, is_test=False)

        args = TrainingArguments(
            output_dir=f"/kaggle/working/deberta_fold_{fold}",
            num_train_epochs=DEBERTA_EPOCHS,
            per_device_train_batch_size=DEBERTA_BATCH,
            per_device_eval_batch_size=DEBERTA_BATCH,
            learning_rate=DEBERTA_LR,
            weight_decay=0.01,
            warmup_ratio=0.06,
            lr_scheduler_type="cosine",
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=100,
            fp16=USE_FP16,
            dataloader_num_workers=0,
            report_to="none",
            seed=SEED + fold,
            use_cpu=not USE_CUDA,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=collator,
        )
        trainer.train()
        val_pred = trainer.predict(val_ds)
        val_probs = softmax_np(val_pred.predictions)
        deberta_oof[va_idx] = val_probs
        print(f"    val log_loss={log_loss(fold_val['label'].values, val_probs):.5f}")

        @torch.no_grad()
        def predict_frame(frame: pd.DataFrame, batch_size: int = 8) -> np.ndarray:
            model.eval()
            device = next(model.parameters()).device
            probs = np.zeros((len(frame), 3), dtype=np.float64)
            for start in range(0, len(frame), batch_size):
                batch = frame.iloc[start : start + batch_size]
                texts = [
                    build_pair_text(r.prompt, r.response_a, r.response_b)
                    for r in batch.itertuples(index=False)
                ]
                enc = tokenizer(
                    texts,
                    max_length=MAX_LEN,
                    truncation=True,
                    padding=True,
                    return_tensors="pt",
                )
                enc = {k: v.to(device) for k, v in enc.items()}
                logits = model(**enc).logits.float().cpu().numpy()
                probs[start : start + len(batch)] = softmax_np(logits)
            return probs

        fold_test_probs = predict_frame(test_df, batch_size=DEBERTA_BATCH)
        if SWAP_AVERAGE_INFERENCE:
            swapped_test = test_df.copy()
            swapped_test["response_a"] = test_df["response_b"].values
            swapped_test["response_b"] = test_df["response_a"].values
            fold_test_sw = predict_frame(swapped_test, batch_size=DEBERTA_BATCH)
            fold_test_probs = average_with_swap(fold_test_probs, fold_test_sw)

        deberta_test_probs += fold_test_probs / DEBERTA_FOLDS
        del model, trainer
        if USE_CUDA:
            torch.cuda.empty_cache()

    print(f"[log] deberta OOF log_loss={log_loss(deberta_train_df['label'].values, deberta_oof):.5f}")
except Exception as exc:
    deberta_ok = False
    deberta_test_probs = tfidf_probs.copy()
    print(f"[warn] DeBERTa stage failed ({exc}); using TF-IDF predictions as DeBERTa branch")

# ── Soft-voting hybrid ensemble ──────────────────────────────────────────────
print("\n[stage] soft-voting ensemble")
w_deberta = ENSEMBLE_WEIGHTS["deberta"]
w_tfidf = ENSEMBLE_WEIGHTS["tfidf"]
final_probs = w_deberta * deberta_test_probs + w_tfidf * tfidf_probs
final_probs = final_probs / final_probs.sum(axis=1, keepdims=True)

id_to_probs = {str(tid): p for tid, p in zip(test_df["id"].values, final_probs)}
sub = sample_sub.copy()
sub["id"] = sub["id"].astype(str)
sub["winner_model_a"] = sub["id"].map(lambda i: float(id_to_probs[i][0]))
sub["winner_model_b"] = sub["id"].map(lambda i: float(id_to_probs[i][1]))
sub["winner_tie"] = sub["id"].map(lambda i: float(id_to_probs[i][2]))

row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print("[check] row_sum_mean", float(row_sums.mean()))
if sub[["winner_model_a", "winner_model_b", "winner_tie"]].isnull().any().any():
    raise ValueError("Missing predictions in submission")

sub.to_csv(OUTPUT_SUBMISSION, index=False)
print("[log] wrote", OUTPUT_SUBMISSION)
print(sub.head(10).to_string(index=False))
